# Initialize Prefect variable `processing-storage-configuration`

In [ ]:
# Imports
import json
import os
import shutil
from IPython.display import Markdown
from pathlib import Path
from prefect.variables import Variable
from resources import utils

In [ ]:
if not utils.cluster_mode:
    raise RuntimeError("This notebook should only be run in cluster mode!")

In [ ]:
# TEMP
from importlib import reload
reload(utils)
utils.init_demo()

In [ ]:
var_name = "processing-storage-configuration"
try:
    existing_values = await Variable.get(var_name) #, _sync=True)
    dump = f"\n{json.dumps(dict(sorted(existing_values.items())), indent=2)}"
except AttributeError:
    existing_values = {}
    dump = "None"
print(f"Existing values for {var_name!r}: {dump}")

In [ ]:
# For the new values, start from the template file that is bundled with rs-client-libraries. 
# We copy it to the local folder with a .txt extension to open it with the text editor.
import rs_client
template_file_org = "storage_configuration.json"
template_file = template_file_org + ".txt"
shutil.copy(
    str(Path(rs_client.__file__).parent.parent / "config" / template_file_org), 
    template_file
)
!chmod u+w ./{template_file}

# Then display a markdown message with a link to it
relative_from_home = str(Path(template_file).resolve().relative_to(Path.home(), walk_up=True))
display(Markdown(f"""Open and modify template file: [{relative_from_home}]({template_file})

Then run the next cell."""),
)

In [ ]:
# Read the modified template file
with open(str(template_file), encoding="utf-8") as f:
    new_values = json.load(f)

print(f"New values for {var_name!r}:\n{json.dumps(new_values, indent=2)}")

In [ ]:
# Overwrite values ?

print(utils.dict_diff(existing_values, new_values))
answer = input(f"\nOverwrite these values in Prefect block {var_name!r} (Y/n)?")

if answer.lower() == "y":
    print(f"Overwriting {var_name!r}")
    await Secret(value=new_values).save("env-vars", overwrite=True)

In [ ]:
!pip install pytest

In [ ]:
from _pytest.config import get_config
from _pytest.assertion import pytest_assertrepr_compare

config = get_config()
config.parse(["-v"])

from _pytest.terminal import TerminalReporter
import sys
reporter = TerminalReporter(config, sys.stdout)
config.pluginmanager.register(reporter, 'terminalreporter')

from collections import OrderedDict

def order_dict(dictionary):
    """https://stackoverflow.com/a/47882384"""
    result = {}
    for k, v in sorted(dictionary.items()):
        if isinstance(v, dict):
            result[k] = order_dict(v)
        elif isinstance(v, list):
            result[k] = [order_dict(v2) for v2 in v]
        else:
            result[k] = v
    return result

def order_dict(obj):
    if isinstance(obj, dict):
        return {
            key: order_dict(value)
            for key, value in sorted(obj.items())
        }
    elif isinstance(obj, list):
        return [order_dict(item) for item in obj]
    else:
        return obj

lines = pytest_assertrepr_compare(config, "==", order_dict(new_values), order_dict(existing_values))

In [ ]:
existing_values == new_values

In [ ]:
if lines:
    print("\n".join(lines))

In [ ]:
new_values

In [ ]:
existing_values["product"]["specific"]

In [ ]:
order_dict(existing_values["product"]["specific"])

In [ ]:
new_values["product"]["specific"]

In [ ]:
existing_values["product"]["specific"] == new_values["product"]["specific"]

In [ ]:


config.pluginmanager.has_plugin("terminalreporter")